In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeDocBS
from hedging.plot_utils import plot_portfolio_vs_option_price
from torch.distributions import LogNormal
from torchrl.modules import TanhNormal
from torchrl.envs import GymWrapper
from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.modules import ProbabilisticActor, SafeModule
from tensordict.nn import TensorDictModule
from torchrl.objectives import ClipPPOLoss
from torchrl.modules import ProbabilisticActor, SafeModule
from torchrl.modules import (
    ValueOperator,
    ActorValueOperator,
    NormalParamExtractor,
)
from torchrl.objectives.value import GAE
from torchrl.envs.utils import ExplorationType, set_exploration_type

/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


# PPO (Recurrent)

In [2]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
H = np.array([[42.5, 45.0], [85.0, 90.0], [170.0, 180.0]])  # barrier
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])

num_paths = 32
num_steps = 250
history_len = 5

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, num_paths, num_steps, history_len=history_len)
env = GymWrapper(base_env)

action_dim = 2

In [24]:
env.action_spec.shape[-1]

2

In [3]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

(48000, 4800)

In [4]:
# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

In [5]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.rnn = nn.LSTM(
            input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.0
        )

    def forward(self, x):
        if len(x.shape) > 3:  # Handle 4D input
            x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
            output, _ = self.rnn(x_reshaped)
            # Reshape output back to original batch dimensions
            output = output.view(
                x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
            )
        else:  # Handle 3D input directly
            output, _ = self.rnn(x)
        output = output[..., -1, :]  # Take output from the last time step
        return output

feature_extractor = SafeModule(
    module=FeatureExtractor(),
    in_keys=["observation"],
    out_keys=["feature"],
)
policy_network = TensorDictModule(
    nn.Sequential(torch.nn.Linear(64, 2*action_dim), NormalParamExtractor()),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)
actor = ProbabilisticActor(
    module=policy_network,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=TanhNormal,
    return_log_prob=True,
)
critic = ValueOperator(
    module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
    in_keys=["feature"],
    out_keys=["state_value"],
)
model = ActorValueOperator(feature_extractor, actor, critic)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ActorValueOperator(
    module=ModuleList(
      (0): SafeModule(
          module=FeatureExtractor(
            (rnn): LSTM(17, 64, num_layers=2, batch_first=True)
          ),
          device=cpu,
          in_keys=['observation'],
          out_keys=['feature'])
      (1): ProbabilisticActor(
          module=ModuleList(
            (0): TensorDictModule(
                module=Sequential(
                  (0): Linear(in_features=64, out_features=4, bias=True)
                  (1): NormalParamExtractor(
                    (scale_mapping): biased_softplus()
                  )
                ),
                device=cpu,
                in_keys=['feature'],
                out_keys=['loc', 'scale'])
            (1): SafeProbabilisticModule(
                in_keys=['loc', 'scale'],
                out_keys=['action', 'action_log_prob'],
                distribution_class=<class 'torchrl.modules.distributions.continuous.TanhNormal'>, 
                distribution_kwargs={}),
   

In [7]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)

In [8]:
loss_module = ClipPPOLoss(
    actor_network=model.get_policy_operator(),
    critic_network=model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

/Users/manu13/Desktop/PHD/DeepHedging/.DeepHedging/lib/python3.11/site-packages/torchrl/objectives/ppo.py:511: DeprecationWarning: 'entropy_coef' is deprecated and will be removed in torchrl v0.11. Please use 'entropy_coeff' instead.
  warnings.warn(


In [9]:
optim = torch.optim.Adam(loss_module.parameters(),lr=5e-5)

In [10]:
num_epochs = 10
num_episodes = 100

In [11]:
for epoch in range(num_epochs):
    for episode in range(num_episodes):
        env.reset(seed=epoch + 1000)
        collector = SyncDataCollector(
            env,
            model.get_policy_operator(),
            frames_per_batch=frames_per_batch,
            total_frames=frames_per_batch,
            device=device,
        )
        replay_buffer = ReplayBuffer(
            storage=LazyTensorStorage(max_size=frames_per_batch),
            sampler=SamplerWithoutReplacement(),
        )
        for batch in collector:
            advantage_module(batch)
            replay_buffer.extend(batch.reshape(-1).cpu())
            for _ in range(sub_batch_num):
                subdata = replay_buffer.sample(sub_batch_size)
                optim.zero_grad()
                # Forward pass PPO loss
                loss = loss_module(subdata.to(device))
                loss_critic, loss_objective, loss_entropy = (
                    loss["loss_critic"],
                    loss["loss_objective"],
                    loss["loss_entropy"],
                )
                loss_sum = loss_critic + loss_objective + loss_entropy
                # Backward pass
                loss_sum.backward()
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_norm=1.0)
                for param in loss_module.parameters():
                    if param.grad is not None:
                        param.grad = torch.nan_to_num(param.grad)
                # Update the networks
                optim.step()

        if (episode + 1) % 10 == 0:
            print(
                f"""Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss_sum.item()}, Loss Critic: {loss_critic.item()}, Loss Obj. {loss_objective.item()}, Loss Ent. {loss_entropy.item()}, Avg. Reward: {batch['next', 'reward'].mean().item()}"""
            )

Epoch 1/10, Episode 10/100, Loss: 248.82205200195312, Loss Critic: 124.15958404541016, Loss Obj. 124.66378784179688, Loss Ent. -0.0013152648461982608, Avg. Reward: -7.635416507720947


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'ppo_rnn.pth')

In [ ]:
# Test

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=model.get_policy_operator())

In [ ]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

In [ ]:
plot_portfolio_vs_option_price(env._env)

## PPO (Recurrent): Risk-Adjusted Return Reward

In [ ]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
H = np.array([[42.5, 45.0], [85.0, 90.0], [170.0, 180.0]])  # barrier
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])

num_paths = 32
num_steps = 250
history_len = 5

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, num_paths, num_steps, history_len=history_len, reward_type="risk_adjusted_return")
env = GymWrapper(base_env)

action_dim = 2

In [ ]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

In [ ]:
# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.rnn = nn.LSTM(
            input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.0
        )

    def forward(self, x):
        if len(x.shape) > 3:  # Handle 4D input
            x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
            output, _ = self.rnn(x_reshaped)
            # Reshape output back to original batch dimensions
            output = output.view(
                x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
            )
        else:  # Handle 3D input directly
            output, _ = self.rnn(x)
        output = output[..., -1, :]  # Take output from the last time step
        return output

feature_extractor = SafeModule(
    module=FeatureExtractor(),
    in_keys=["observation"],
    out_keys=["feature"],
)
policy_network = TensorDictModule(
    nn.Sequential(torch.nn.Linear(64, 2*action_dim), NormalParamExtractor()),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)
actor = ProbabilisticActor(
    module=policy_network,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=TanhNormal,
    return_log_prob=True,
)
critic = ValueOperator(
    module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
    in_keys=["feature"],
    out_keys=["state_value"],
)
model = ActorValueOperator(feature_extractor, actor, critic)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)

In [ ]:
loss_module = ClipPPOLoss(
    actor_network=model.get_policy_operator(),
    critic_network=model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

In [ ]:
optim = torch.optim.Adam(loss_module.parameters(),lr=5e-5)

In [ ]:
num_epochs = 10
num_episodes = 100

In [ ]:
# Train

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        env.reset(seed=epoch + 1000)
        collector = SyncDataCollector(
            env,
            model.get_policy_operator(),
            frames_per_batch=frames_per_batch,
            total_frames=frames_per_batch,
            device=device,
        )
        replay_buffer = ReplayBuffer(
            storage=LazyTensorStorage(max_size=frames_per_batch),
            sampler=SamplerWithoutReplacement(),
        )
        for batch in collector:
            advantage_module(batch)
            replay_buffer.extend(batch.reshape(-1).cpu())
            for _ in range(sub_batch_num):
                subdata = replay_buffer.sample(sub_batch_size)
                optim.zero_grad()
                # Forward pass PPO loss
                loss = loss_module(subdata.to(device))
                loss_critic, loss_objective, loss_entropy = (
                    loss["loss_critic"],
                    loss["loss_objective"],
                    loss["loss_entropy"],
                )
                loss_sum = loss_critic + loss_objective + loss_entropy
                # Backward pass
                loss_sum.backward()
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_norm=1.0)
                for param in loss_module.parameters():
                    if param.grad is not None:
                        param.grad = torch.nan_to_num(param.grad)
                # Update the networks
                optim.step()

        if (episode + 1) % 10 == 0:
            print(
                f"""Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss_sum.item()}, Loss Critic: {loss_critic.item()}, Loss Obj. {loss_objective.item()}, Loss Ent. {loss_entropy.item()}, Avg. Reward: {batch['next', 'reward'].mean().item()}"""
            )

In [ ]:
torch.save(model.state_dict(), 'ppo_rnn_rar.pth')

In [ ]:
# Test

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=model.get_policy_operator())

In [ ]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

In [ ]:
plot_portfolio_vs_option_price(env._env)

## Test Transaction Costs

In [ ]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
H = np.array([[42.5, 45.0], [85.0, 90.0], [170.0, 180.0]])  # barrier
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])

num_paths = 32
num_steps = 250
history_len = 5
transaction_cost = True
transaction_fee_rate = 0.1 # Test high transaction cost

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate)
env = GymWrapper(base_env)

action_dim = 2

In [ ]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

In [ ]:
# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.rnn = nn.LSTM(
            input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.0
        )

    def forward(self, x):
        if len(x.shape) > 3:  # Handle 4D input
            x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
            output, _ = self.rnn(x_reshaped)
            # Reshape output back to original batch dimensions
            output = output.view(
                x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
            )
        else:  # Handle 3D input directly
            output, _ = self.rnn(x)
        output = output[..., -1, :]  # Take output from the last time step
        return output

feature_extractor = SafeModule(
    module=FeatureExtractor(),
    in_keys=["observation"],
    out_keys=["feature"],
)
policy_network = TensorDictModule(
    nn.Sequential(torch.nn.Linear(64, 2*action_dim), NormalParamExtractor()),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)
actor = ProbabilisticActor(
    module=policy_network,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=TanhNormal,
    return_log_prob=True,
)
critic = ValueOperator(
    module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
    in_keys=["feature"],
    out_keys=["state_value"],
)
model = ActorValueOperator(feature_extractor, actor, critic)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)

In [ ]:
loss_module = ClipPPOLoss(
    actor_network=model.get_policy_operator(),
    critic_network=model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

In [ ]:
optim = torch.optim.Adam(loss_module.parameters(),lr=5e-5)
num_epochs = 10
num_episodes = 100

In [ ]:
# Train

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        env.reset(seed=epoch + 1000)
        collector = SyncDataCollector(
            env,
            model.get_policy_operator(),
            frames_per_batch=frames_per_batch,
            total_frames=frames_per_batch,
            device=device,
        )
        replay_buffer = ReplayBuffer(
            storage=LazyTensorStorage(max_size=frames_per_batch),
            sampler=SamplerWithoutReplacement(),
        )
        for batch in collector:
            advantage_module(batch)
            replay_buffer.extend(batch.reshape(-1).cpu())
            for _ in range(sub_batch_num):
                subdata = replay_buffer.sample(sub_batch_size)
                optim.zero_grad()
                # Forward pass PPO loss
                loss = loss_module(subdata.to(device))
                loss_critic, loss_objective, loss_entropy = (
                    loss["loss_critic"],
                    loss["loss_objective"],
                    loss["loss_entropy"],
                )
                loss_sum = loss_critic + loss_objective + loss_entropy
                # Backward pass
                loss_sum.backward()
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_norm=1.0)
                for param in loss_module.parameters():
                    if param.grad is not None:
                        param.grad = torch.nan_to_num(param.grad)
                # Update the networks
                optim.step()

        if (episode + 1) % 10 == 0:
            print(
                f"""Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss_sum.item()}, Loss Critic: {loss_critic.item()}, Loss Obj. {loss_objective.item()}, Loss Ent. {loss_entropy.item()}, Avg. Reward: {batch['next', 'reward'].mean().item()}"""
            )

In [ ]:
# Test

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, 5, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=model.get_policy_operator())

In [ ]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

In [ ]:
plot_portfolio_vs_option_price(env._env)

## Adapt LSTM to Handle Transaction Costs

Introduce a parameter that decides whether we choose to update the portfolio or not.  
If we choose **not to update** the portfolio, we must ensure:

```python
positions[..., 0] == self.call_held[..., self.current_step]
positions[..., 1] == self.put_held[..., self.current_step]
```

Where:

```python
action = action.reshape(-1, self.action_dim)

assert self.action_space.contains(action), f"{action!r} ({type(action)}) invalid"  # Getting error here with MLP
assert self.state is not None, "Call reset before using step method."

# Reshape actions to match our dimensions
action_reshaped = action.reshape(
    self.num_paths, self.num_assets, self.num_strikes, self.action_dim
)
```

And:

```python
positions = action_reshaped
```



In [ ]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
H = np.array([[42.5, 45.0], [85.0, 90.0], [170.0, 180.0]])  # barrier
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])

num_paths = 32
num_steps = 250
history_len = 5
transaction_cost = True
transaction_fee_rate = 0.1 # Test high transaction cost

base_env = HedgeDocBS(S0, K, H, maturity, r, sigma, num_paths, num_steps, history_len=history_len, 
                      transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate)
env = GymWrapper(base_env)

action_dim = 2

In [ ]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

In [ ]:
# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.rnn = nn.LSTM(
            input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.0
        )

    def forward(self, x):
        if len(x.shape) > 3:  # Handle 4D input
            x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
            output, _ = self.rnn(x_reshaped)
            # Reshape output back to original batch dimensions
            output = output.view(
                x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
            )
        else:  # Handle 3D input directly
            output, _ = self.rnn(x)
        output = output[..., -1, :]  # Take output from the last time step
        return output

feature_extractor = SafeModule(
    module=FeatureExtractor(),
    in_keys=["observation"],
    out_keys=["feature"],
)
policy_network = TensorDictModule(
    nn.Sequential(torch.nn.Linear(64, 2*action_dim), NormalParamExtractor()),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)
actor = ProbabilisticActor(
    module=policy_network,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=TanhNormal,
    return_log_prob=True,
)
critic = ValueOperator(
    module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
    in_keys=["feature"],
    out_keys=["state_value"],
)
model = ActorValueOperator(feature_extractor, actor, critic)